Install necessary libraries

In [ ]:
!pip install -qU transformers datasets peft bitsandbytes accelerate sacrebleu evaluate gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 70.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.4/485.4 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 MB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.1/322.1 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:


import os
import random
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset, Dataset, concatenate_datasets
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import (
    prepare_model_for_kbit_training,
    LoraConfig,
    get_peft_model,
    PeftModel,
    PeftConfig, TaskType, AutoPeftModelForCausalLM
)



from google.colab import files

from functools import lru_cache  # Import lru_cache from functools
import gradio as gr


In [ ]:
HF_API_TOKEN = userdata.get('HF_TOKEN')
os.environ["HF_TOKEN"] = HF_API_TOKEN

In [ ]:
# Global variables for model and tokenizer
model = None
tokenizer = None

Load the peft model from HuggingFace

In [ ]:
# Cache loading of model to avoid reloading for each inference
@lru_cache(maxsize=1)
def load_model():
    peft_model_path = 'vbanwari/Fine-tuned-for-language-translation-bloomz3b-tatoeba'

    #Load Base Model (BLOOMZ-3b) with 4-bit Quantization
    model_name = "bigscience/bloomz-3b"

    # Configure 4-bit quantization
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16
    )

    # Load the tokenizer
    tokenizer = AutoTokenizer.from_pretrained(peft_model_path)

    # Set the padding token to be the same as the end-of-sequence token
    tokenizer.pad_token = tokenizer.eos_token

    # Specify that padding should be added to the right side of the sequences
    tokenizer.padding_side = "right"

    base_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.float16
    )

    tuned_model = PeftModel.from_pretrained(base_model, peft_model_path)



    # Enable attention cache during inference
    tuned_model.config.use_cache = True

    tuned_model.eval()

    return tuned_model, tokenizer

Translate/evaluate function

In [ ]:
def translate(german_text):
    if not german_text or german_text.strip() == "":
        return "Please enter some German text to translate."

    # Load model if not already loaded
    global model, tokenizer
    if model is None or tokenizer is None:
        try:
            model, tokenizer = load_model()
        except Exception as e:
            return f"Error loading model: {str(e)}"

    # Create prompt
    prompt = f"Instruction: Translate the following German sentence into French.\nGerman: {german_text}\nFrench:"

    # Generate translation
    try:
        # Tokenize and send to the same device as the model
        inputs = tokenizer(prompt, return_tensors="pt")
        # Place input on the same device as the model's first parameter
        for k, v in inputs.items():
            # Get the device of the first parameter in the model
            for param in model.parameters():
                device = param.device
                break
            inputs[k] = v.to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=100,
                num_beams=5,
                temperature=0.3,
                early_stopping=True
            )

        full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
        french_text = full_output.split("French:")[-1].strip() if "French:" in full_output else full_output

        return french_text
    except Exception as e:
        return f"Translation error: {str(e)}"

Create and launch the Gradio interface

In [ ]:
# Create Gradio interface
with gr.Blocks() as demo:
    gr.Markdown("# 🌐 German to French Translation")

    with gr.Row():
        with gr.Column():
            german_input = gr.Textbox(label="German Text", lines=4)
            translate_btn = gr.Button("Translate to French")

        with gr.Column():
            french_output = gr.Textbox(label="French Translation", lines=4)

    # Examples
    gr.Examples(
        examples=[
            ["Ich liebe es, neue Sprachen zu lernen."],
            ["Wie viel kostet ein Ticket nach Paris?"],
            ["Kannst du mir bitte helfen?"]
        ],
        inputs=german_input
    )

    # Set up events
    translate_btn.click(
        fn=translate,
        inputs=german_input,
        outputs=french_output
    )

# Launch app
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://069f7cf90388602d41.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
